# Pixel ridge · 02 公式检查、Gaussian MC 与真实 LOOCV

默认 power-law input covariance, d=128,n=256，便于交互。可改 DATASET 为 isotropic、vanhateren 或 ffhq；后两者使用完整实测谱，并不会偷偷截取 top PCs。全尺寸 MC 请先检查 pilot ETA 与内存需求。

MC 回归输入为 Gaussian pixel vectors，坐标采用 population PCs；没有特征映射。固定 λ、DE_gen_CV、DE_acc_oracle 使用理论给出的固定 alpha；额外 empirical_RidgeCV 则每个 trial 真正根据 LOOCV 误差选 alpha。LOOCV noisy-response risk 比 noiseless E_gen 多一个常数 σ²，在期望目标上有相同最优点，但选参本身有随机性。

所有 MC policy 共用数据与噪声。ACC oracle 由 DE 选参，未用报告的 MC 误差挑最优 trial 或 alpha。


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, time
os.environ.setdefault("MPLCONFIGDIR", "/tmp/accentuationpredrmt-matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/accentuationpredrmt-xdg-cache")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"rmt_core").is_dir())
sys.path.insert(0, str(ROOT))
from scripts import pixel_ridge_notebook_utils as u
threads = threadpool_limits(limits=2)
OUT = ROOT/"notebooks/outputs/pixel_ridge"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# vanhateren / ffhq: full measured input spectrum, no top-PC truncation.
# powerlaw / isotropic: synthetic pixel covariance; D controls dimension.
DATASET = "powerlaw"
D, N = 128, 256
SEED = 42
RATIOS = np.r_[0., np.geomspace(1e-6, 10., 9)]
ALPHAS = np.logspace(-5, 7, 181)  # alpha = n * lambda
FIXED_LAMBDA = 0.1
ACC_OBJECTIVE = "E_acc"          # or "E_acc_corrected"
MAX_SECONDS = 180
FORCE = False
s, beta = u.load_problem(ROOT, DATASET, D, SEED)
S = float(s @ beta**2)
print("d =", len(s), "n =", N, "signal variance =", S)


In [ ]:
config = dict(dataset=DATASET, d=len(s), n=N, seed=SEED,
              ratios=RATIOS.tolist(), alphas=ALPHAS.tolist(),
              fixed_lambda=FIXED_LAMBDA, acc_objective=ACC_OBJECTIVE)
digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())
digest.update(s.tobytes()); digest.update(beta.tobytes())
for path in [Path(u.__file__), *sorted((ROOT/"rmt_core").glob("*.py"))]:
    digest.update(path.read_bytes())
run_dir = OUT/digest.hexdigest()[:16]
run_dir.mkdir(parents=True, exist_ok=True)
LOG = run_dir/"progress.log"
def log(message):
    print(message, flush=True)
    with LOG.open("a") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S") + " | " + message + "\n")
log("Cache/log directory: " + str(run_dir))
if (run_dir/"summary.csv").exists() and not FORCE:
    summary = pd.read_csv(run_dir/"summary.csv")
    paths = pd.read_csv(run_dir/"paths.csv")
    log("Loaded plot-ready tables.")
else:
    start = time.perf_counter()
    u.sweep(s, beta, N, RATIOS[-1:], ALPHAS, FIXED_LAMBDA, ACC_OBJECTIVE, progress=False)
    eta = (time.perf_counter()-start)*len(RATIOS)
    log(f"One-noise pilot ETA: {eta:.1f}s")
    if eta > MAX_SECONDS:
        raise RuntimeError("Projected time exceeds MAX_SECONDS; adjust grid or budget.")
    summary, paths = u.sweep(s, beta, N, RATIOS, ALPHAS, FIXED_LAMBDA, ACC_OBJECTIVE)
    summary.to_csv(run_dir/"summary.csv", index=False)
    paths.to_csv(run_dir/"paths.csv", index=False)
    log("DE sweep complete.")
(run_dir/"config.json").write_text(json.dumps(config, indent=2))
display(summary.head())
display(summary[summary.boundary == True][["policy","ratio","alpha"]])


## 逐项检查 DE

设 s 是 Σ 特征值，β 是 teacher 的 PC coefficients，q_k=s_k/(s_k+κ)。
m_k=q_k β_k；v_k=[κ² Σ_j s_j β_j²/(s_j+κ)²+σ²]/[n−df₂] · s_k/(s_k+κ)²。

E_gen=Σ_k s_k[(m_k−β_k)²+v_k]；
μN=βᵀm，μD=Σ_k(m_k²+v_k)。
λ=κ[1−df₁(κ)/n]，alpha=nλ。
κ 的可行域受此方程限制；因此代码在正 alpha 上优化并解 physical κ branch。


In [ ]:
row = summary[(summary.policy=="DE_acc_oracle") & (summary.ratio==RATIOS[-1])].iloc[0]
k, sigma = row.kappa, row.sigma
shrink = s/(s+k)
m = shrink*beta
df2 = np.sum(shrink**2)
v = ((k*k*np.sum(s*beta**2/(s+k)**2)+sigma**2)/(N-df2))*s/(s+k)**2
mu_N, mu_D = beta@m, np.sum(m*m+v)
Eg = np.sum(s*((m-beta)**2+v))/S
Ea = (1-mu_N/mu_D)**2
assert np.allclose([Eg, Ea], [row.E_gen, row.E_acc])
assert np.isclose(row.lam, k*(1-shrink.sum()/N))
display(pd.Series(dict(E_gen=Eg,E_acc=Ea,mu_N=mu_N,mu_D=mu_D,lambda_=row.lam,kappa=k)))


## 实际 LOOCV 与缓存 MC

零均值 Gaussian 数据采用 fit_intercept=False。SVD 精确计算 hat diagonal，LOO residual=(y−Hy)/(1−diag H)。CV 候选为 ALPHAS；理论数值细化得到的 alpha 也用于固定策略，但不额外加入 empirical CV 的候选。

N_TRIALS、SEED 改变后使用新缓存。零点附近误差请看均值/分布，不能由 leading-DE 的零直接推断真实 MC 零。


In [ ]:
N_TRIALS = 10
MC_SEED = 17
RUN_MC = True
mc_path = run_dir/f"mc_trials{N_TRIALS}_seed{MC_SEED}.csv"
mc = None
if RUN_MC:
    if mc_path.exists() and not FORCE:
        mc = pd.read_csv(mc_path)
        log("Loaded MC cache.")
    else:
        # The pilot includes all noise settings and policies.
        estimated_bytes = 8*(N*len(s) + min(N,len(s))*(N+len(s))
                             + len(s)*(len(ALPHAS)+3))
        log(f"Main-array lower-bound memory estimate: {estimated_bytes/1e6:.1f} MB")
        start = time.perf_counter()
        u.monte_carlo(s,beta,N,summary,ALPHAS,1,MC_SEED+100000,False)
        eta = (time.perf_counter()-start)*N_TRIALS
        log(f"MC pilot ETA {eta:.1f}s, {N_TRIALS} trials")
        if eta > MAX_SECONDS:
            raise RuntimeError("Reduce workload or deliberately raise MAX_SECONDS.")
        mc = u.monte_carlo(s,beta,N,summary,ALPHAS,N_TRIALS,MC_SEED)
        mc.to_csv(mc_path,index=False)
        log("MC complete: "+str(mc_path))


In [ ]:
fig, axes = u.plot_policies(summary)
if mc is not None:
    means = mc.groupby(["policy","ratio"],as_index=False).mean(numeric_only=True)
    for j, policy in enumerate(["fixed","DE_gen_CV","DE_acc_oracle"]):
        part = means[means.policy==policy].sort_values("sigma")
        for metric, color in [("E_gen","C0"),("E_acc","C1")]:
            axes[0,j].plot(part.sigma,np.maximum(part[metric],1e-16),"o",color=color,ms=4)
    cv = means[means.policy=="empirical_RidgeCV"].sort_values("sigma")
    axes[0,1].plot(cv.sigma,np.maximum(cv.E_gen,1e-16),"x:",color="C0",label="actual CV Egen")
    axes[0,1].plot(cv.sigma,np.maximum(cv.E_acc,1e-16),"x:",color="C1",label="actual CV Eacc")
    axes[1,1].plot(cv.sigma,cv.lam,"x:",color="C3",label="MC mean CV lambda")
    axes[0,1].legend(fontsize=7); axes[1,1].legend(fontsize=7)
fig.savefig(run_dir/"de_mc.png",dpi=180,bbox_inches="tight")
plt.show()


## 选参变动、均值/中位数与边界

E_acc oracle 优化的是理论近似，不是 empirical MC oracle；其 leading 值可能接近零而 MC 均值非零。增大 trials、切换 ACC_OBJECTIVE，检查两者差异。二阶修正仍缺少完整 design fluctuations。


In [ ]:
if mc is not None:
    display(mc.groupby(["policy","ratio"])[["E_gen","E_acc","alpha"]].agg(["mean","median","std"]))
    cv = mc[mc.policy=="empirical_RidgeCV"]
    print("CV boundary fraction:", np.mean((cv.alpha==ALPHAS[0]) | (cv.alpha==ALPHAS[-1])))
display(summary[["policy","ratio","alpha","kappa","E_acc","E_acc_corrected","boundary"]])


## 4. Distributional-DE oracle：匹配矩的 Gaussian 权重近似

这不是已经证明的完整权重分布定理。我们尝试
\[
\hat\beta_G=m_\kappa+\operatorname{diag}(v_\kappa)^{1/2}\xi,\quad
m_j=\frac{s_j}{s_j+\kappa}\beta_j^*,\quad
v_j=\frac{E_{\rm gen}+\sigma^2}{n}\frac{s_j}{(s_j+\kappa)^2},\quad\xi\sim N(0,I).
\]
这里公式中的 E_gen 是未归一化的风险。此近似匹配 DE 的逐坐标二阶矩，但忽略尚未验证的非对角协方差和高阶累积量；不能仅凭匹配矩就断言比值分布正确。

用同一个权重样本计算 N=β*ᵀβ̂ 和 D=‖β̂‖²，优化样本平均 (1−N/D)²。共同随机数让正则化路径更平滑；独立 surrogate draws 检验搜索过拟合；独立实际 ridge 拟合检验 surrogate 偏差。独立 surrogate 标准误仅反映数值积分误差。所有选择都依赖 teacher，是 oracle，不是可直接用于真实未知 teacher 的 CV。

**重要**：E_acc 小不必然意味着 R²_acc 的期望稳定。后者涉及 D/N；N 接近零时可能有重尾甚至不存在有限期望。当前只优化 E_acc。


In [ ]:
RUN_DISTRIBUTIONAL = True
G_DRAWS, G_VALIDATION = 1024, 4096
G_SEED = 123
G_ALPHAS = np.logspace(np.log10(ALPHAS[0]), np.log10(ALPHAS[-1]), 81)
# Independent knobs: start small; full natural-image dimensions are costlier.
G_RATIOS = RATIOS[np.unique(np.linspace(1, len(RATIOS)-1, 4).astype(int))]
g_config = dict(draws=G_DRAWS, validation=G_VALIDATION, seed=G_SEED,
                alphas=G_ALPHAS.tolist(), ratios=G_RATIOS.tolist())
g_tag = hashlib.sha256(json.dumps(g_config, sort_keys=True).encode()).hexdigest()[:12]
g_dir = run_dir / ("gaussian_" + g_tag)
g_dir.mkdir(exist_ok=True)
if RUN_DISTRIBUTIONAL:
    if (g_dir/"summary.csv").exists() and not FORCE:
        gaussian = pd.read_csv(g_dir/"summary.csv")
        gaussian_paths = pd.read_csv(g_dir/"paths.csv")
    else:
        memory_mb = 8 * len(s) * (G_DRAWS + G_VALIDATION + 3*max(G_DRAWS,G_VALIDATION))/1e6
        log(f"Gaussian surrogate main-array memory estimate: {memory_mb:.1f} MB")
        if memory_mb > 1024:
            raise RuntimeError("Reduce draws/dimension before allocating >1 GB; inspect memory first.")
        started = time.perf_counter()
        u.gaussian_acc_selection(s,beta,N,G_RATIOS[-1:],G_ALPHAS,
            G_DRAWS,G_VALIDATION,G_SEED,progress=False)
        eta = (time.perf_counter()-started)*len(G_RATIOS)
        log(f"Gaussian selection pilot ETA: {eta:.1f}s")
        if eta > MAX_SECONDS:
            raise RuntimeError("Gaussian pilot exceeds budget; reduce grid/draws or profile.")
        gaussian, gaussian_paths = u.gaussian_acc_selection(
            s,beta,N,G_RATIOS,G_ALPHAS,G_DRAWS,G_VALIDATION,G_SEED)
        gaussian.to_csv(g_dir/"summary.csv",index=False)
        gaussian_paths.to_csv(g_dir/"paths.csv",index=False)
        (g_dir/"config.json").write_text(json.dumps(g_config,indent=2))
    display(gaussian[["ratio","lam","kappa","E_acc","gaussian_selection_risk",
                     "gaussian_validation_risk","gaussian_validation_se","boundary"]])


In [ ]:
if RUN_DISTRIBUTIONAL:
    # Add the new policy to the three existing policies; same fresh MC data
    # across policies. These data were not used to select the Gaussian alpha.
    matched = np.isclose(summary.ratio.to_numpy()[:,None],G_RATIOS[None,:],rtol=1e-10,atol=0).any(axis=1)
    compare = pd.concat([summary[matched],gaussian],ignore_index=True)
    g_mc_path = g_dir / f"ridge_mc_trials{N_TRIALS}_seed{MC_SEED+9000}.csv"
    if g_mc_path.exists() and not FORCE:
        g_mc = pd.read_csv(g_mc_path)
    else:
        started = time.perf_counter()
        u.monte_carlo(s,beta,N,compare,G_ALPHAS,trials=1,
                      seed=MC_SEED+90000,progress=False)
        eta = (time.perf_counter()-started)*N_TRIALS
        log(f"Independent ridge validation pilot ETA: {eta:.1f}s")
        if eta > MAX_SECONDS:
            raise RuntimeError("Independent ridge validation exceeds time budget.")
        g_mc = u.monte_carlo(s,beta,N,compare,G_ALPHAS,trials=N_TRIALS,
                             seed=MC_SEED+9000)
        g_mc.to_csv(g_mc_path,index=False)
    fig, ax = plt.subplots(1,3,figsize=(16,4.5))
    for policy in ["fixed","DE_gen_CV","DE_acc_oracle","Gaussian_acc_oracle"]:
        p = compare[compare.policy==policy].sort_values("sigma")
        c = ax[0].plot(p.sigma,p.lam,".-",label=policy)[0].get_color()
        empirical = g_mc[g_mc.policy==policy].groupby("sigma").E_acc.agg(["mean","sem"])
        ax[1].errorbar(empirical.index,empirical["mean"],yerr=2*empirical["sem"],
                       fmt="o-",color=c,label=policy)
    ax[2].errorbar(gaussian.sigma,gaussian.gaussian_validation_risk,
                   yerr=2*gaussian.gaussian_validation_se,fmt="s--",
                   label="independent Gaussian surrogate")
    empirical = g_mc[g_mc.policy=="Gaussian_acc_oracle"].groupby("sigma").E_acc.agg(["mean","sem"])
    ax[2].errorbar(empirical.index,empirical["mean"],yerr=2*empirical["sem"],
                   fmt="o-",label="actual ridge MC")
    for a,title in zip(ax,["Selected lambda","Actual ridge E acc / S","Gaussian-selected policy: validation"]):
        a.set(xscale="log",yscale="log",xlabel="response noise sigma",title=title)
        top = a.secondary_xaxis("top",functions=(
            lambda x: np.asarray(x)**2/S,lambda x: np.sqrt(np.maximum(x,0)*S)))
        top.set_xlabel("noise / signal variance")
        a.grid(alpha=.2)
        a.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(g_dir/"distributional_oracle_validation.png",dpi=160)
    plt.show()
    log(f"Distributional comparison cached in {g_dir}")
